In [1]:
# 1. Import libraries
import pandas as pd
import re
import nltk
import matplotlib.pyplot as plt

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# 2. Download NLP data
nltk.download('stopwords')
nltk.download('wordnet')

# 3. Load dataset
df = pd.read_csv("bookPublishingData.csv")
print(df.head())
print(df.columns)

# 4. Select text column
# Change 'Description' to your actual text column
text_col = "Description"
df[text_col] = df[text_col].fillna("").astype(str)

# 5. NLP cleaning
stop_words = set(stopwords.words("english"))
lemma = WordNetLemmatizer()

def clean_text(text):
    text = re.sub(r'[^a-zA-Z\s]', '', text.lower())
    words = text.split()
    words = [lemma.lemmatize(w) for w in words if w not in stop_words]
    return " ".join(words)

df["clean_text"] = df[text_col].apply(clean_text)

# 6. TF-IDF
tfidf = TfidfVectorizer(max_features=3000)
X = tfidf.fit_transform(df["clean_text"])

# 7. K-Means
k = 5
model = KMeans(n_clusters=k, random_state=42, n_init=10)
df["Cluster"] = model.fit_predict(X)

# 8. Show cluster keywords
terms = tfidf.get_feature_names_out()

for i in range(k):
    words = model.cluster_centers_[i].argsort()[-10:][::-1]
    print(f"\nCluster {i}:",
          ", ".join(terms[j] for j in words))

# 9. Visualisation
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X.toarray())

plt.figure(figsize=(8,6))
plt.scatter(X_pca[:,0], X_pca[:,1], c=df["Cluster"])
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.title("Book Publishing K-Means Clusters")
plt.show()

# 10. Display results
print(df[["Cluster", text_col]].head(20))

# 11. Save results
df.to_csv("bookPublishingData_clustered.csv", index=False)

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


   Publishing_Year                        Book_Name  \
0           1975.0                          Beowulf   
1           1987.0                 Batman: Year One   
2           2015.0                Go Set a Watchman   
3           2008.0  When You Are Engulfed in Flames   
4           2011.0         Daughter of Smoke & Bone   

                                              Author Language_Code  \
0                             Unknown, Seamus Heaney         en-US   
1  Frank Miller, David Mazzucchelli, Richmond Lew...           eng   
2                                         Harper Lee           eng   
3                                      David Sedaris         en-US   
4                                       Laini Taylor           eng   

  Author_Rating  Book_Average_Rating  Book_Ratings_Count          Genre  \
0        Novice                 3.42              155903  genre fiction   
1  Intermediate                 4.23              145267  genre fiction   
2        Novice        

KeyError: 'Description'